# Example: Building the Sentiment Signal and Rebalancing Engine

In this example, we generate a synthetic market price path, compute the EMA crossover sentiment signal, build the SIM-based allocation context, and run the AI rebalancing engine over a simulated trading year. We compare the adaptive engine's wealth curve to the static min-variance baseline from Session 1.

> **By the end of this example, you will be able to:**
> * Generate a synthetic market index path and compute short/long EMAs
> * Build the lambda sentiment signal and market growth factor from price data
> * Run the rebalancing engine and visualize wealth curves against the Session 1 baseline

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

Define the constants we'll use throughout the notebook — EMA windows, gain, budget, and time parameters.

In [ ]:
let
    # Constants -
    global Δt = 1.0 / 252.0;       # years per trading day
    global N_short = 21;             # short EMA window (~1 month)
    global N_long = 63;              # long EMA window (~1 quarter)
    global N_growth = 10;            # market growth EMA window
    global GAIN = 10.0;              # lambda sensitivity
    global B₀ = 10000.0;            # initial budget ($)
    global offset = N_short + N_long; # warmup period (84 days)
    global n_trading_days = 252;      # 1 year of trading after warmup

    # Asset universe — loosely inspired by a diversified portfolio -
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];

    # SIM parameters (αᵢ, βᵢ, σᵢ) — estimated from "historical" data -
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    println("Constants loaded: offset = $(offset) days, trading period = $(n_trading_days) days")
    println("Tickers: $(tickers)")
end

___
## Task 1: Generate Synthetic Market Path and Compute the Sentiment Signal
We generate a synthetic market index using geometric Brownian motion, then compute the short and long EMAs and the resulting lambda sentiment signal. This follows the same approach as the CHEME-5660 TradeBot — a market-level signal that drives the engine's risk appetite.

> **What should you see?** The price path will show typical random-walk behavior. The short EMA hugs the price closely while the long EMA smooths it. When the short crosses below the long, lambda turns positive (bearish); when it crosses above, lambda turns negative (bullish).

In [ ]:
let
    # Total days needed: warmup + trading period -
    T_total = offset + n_trading_days;

    # Generate synthetic market index (GBM: μ = 8% annual, σ = 18% annual) -
    μ_market = 0.08;
    σ_market = 0.18;
    S₀ = 100.0; # starting price

    global market_prices = zeros(T_total);
    market_prices[1] = S₀;
    for t in 2:T_total
        z = randn();
        market_prices[t] = market_prices[t-1] * exp((μ_market - 0.5 * σ_market^2) * Δt + σ_market * sqrt(Δt) * z);
    end

    # Compute EMAs -
    global ema_short = compute_ema(market_prices; window=N_short);
    global ema_long = compute_ema(market_prices; window=N_long);

    # Compute lambda sentiment signal -
    global lambda_series = compute_lambda(ema_short, ema_long; G=GAIN);

    # Zero out lambda during warmup -
    lambda_series[1:offset] .= 0.0;

    # Compute market growth and smooth with EMA -
    gm_raw = compute_market_growth(market_prices; Δt=Δt);
    global gm_ema = compute_ema(gm_raw; window=N_growth);

    println("Generated $(T_total) days of synthetic market data")
    println("Lambda range: [$(round(minimum(lambda_series), digits=3)), $(round(maximum(lambda_series), digits=3))]")
end

**Visualize:** Market price with EMAs (top) and the lambda sentiment signal (bottom).

> **What should you see?** The top panel shows the synthetic price with short EMA (fast, hugs price) and long EMA (slow, smooth). The bottom panel shows lambda — positive when short < long (bearish), negative when short > long (bullish). The shaded region after the warmup period is where the engine operates.

In [ ]:
let
    T_total = length(market_prices);
    days = 1:T_total;

    # Top: price + EMAs -
    p1 = plot(days, market_prices, label="Price", linewidth=1.5, color=:gray70, alpha=0.8)
    plot!(p1, days, ema_short, label="EMA-$(N_short)", linewidth=2, color=:steelblue)
    plot!(p1, days, ema_long, label="EMA-$(N_long)", linewidth=2, color=:coral)
    vline!(p1, [offset], label="Warmup end", linestyle=:dash, color=:black, alpha=0.5)
    ylabel!(p1, "Price (\$)")
    title!(p1, "Synthetic Market Index with EMAs")

    # Bottom: lambda signal -
    p2 = plot(days, lambda_series, label="λ (sentiment)", linewidth=2, color=:purple)
    hline!(p2, [0.0], label="", linestyle=:dash, color=:black, alpha=0.3)
    vline!(p2, [offset], label="Warmup end", linestyle=:dash, color=:black, alpha=0.5)
    xlabel!(p2, "Trading Day")
    ylabel!(p2, "λ")
    title!(p2, "Sentiment Signal (λ > 0 = bearish, λ < 0 = bullish)")

    plot(p1, p2, layout=(2, 1), size=(800, 550), legend=:topright)
end

___
## Task 2: Build the Context Model and Generate Per-Ticker Synthetic Prices
We construct the full context model the engine needs: a price matrix for all tickers (generated via SIM from the market index), SIM parameters, and the smoothed market growth factor. This mirrors the CHEME-5660 TradeBot's context model setup.

> **What should you see?** Each ticker's synthetic price path will be correlated with the market index through its beta. High-beta assets (SmallCap) will be more volatile and track the market more closely; low/negative-beta assets (Bond) will move independently or inversely.

In [ ]:
let
    T_total = length(market_prices);
    K = length(tickers);

    # Build price matrix: column 1 = day index, columns 2:K+1 = ticker prices -
    global price_matrix = zeros(T_total, K + 1);
    price_matrix[:, 1] = 1:T_total;

    # Starting prices for each ticker -
    start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    # Generate per-ticker prices via SIM: gᵢ = αᵢ + βᵢ × gₘ + σᵢ × √Δt × z -
    gm_raw = compute_market_growth(market_prices; Δt=Δt);

    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
        price_matrix[1, k + 1] = start_prices[ticker];

        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
            price_matrix[t, k + 1] = price_matrix[t - 1, k + 1] * exp(gᵢ);
        end
    end

    # Build the context model -
    global context = build(MyRebalancingContextModel, (
        B = B₀,
        tickers = tickers,
        marketdata = price_matrix,
        marketfactor = gm_ema,
        sim_parameters = sim_params,
        lambda = 0.0,
        Δt = Δt,
        epsilon = 0.1
    ));

    println("Price matrix: $(size(price_matrix)) (days × [index + $(K) tickers])")
    println("Starting prices: $(Dict(t => round(price_matrix[1, k+1], digits=2) for (k, t) in enumerate(tickers)))")
end

**Visualize:** Synthetic per-ticker price paths.

> **What should you see?** Five price paths with varying volatility. SmallCap (β = 1.35) should be most volatile and track the market closely. Bond (β = −0.15) should be relatively flat and sometimes move inversely to the others.

In [ ]:
let
    T_total = size(price_matrix, 1);
    days = 1:T_total;
    colors = [:steelblue :coral :green :goldenrod :purple];

    p = plot(size=(800, 400), title="Synthetic Per-Ticker Price Paths", 
        xlabel="Trading Day", ylabel="Price (\$)")
    for (k, ticker) in enumerate(tickers)
        # normalize to 100 for comparison -
        normalized = price_matrix[:, k+1] ./ price_matrix[1, k+1] .* 100;
        plot!(p, days, normalized, label=ticker, linewidth=1.5, color=colors[k])
    end
    vline!(p, [offset], label="Warmup end", linestyle=:dash, color=:black, alpha=0.5)
    ylabel!(p, "Normalized Price (100 = start)")
    p
end

___
## Task 3: Run the Rebalancing Engine and Compare to Baseline
We define the trigger rules (for now, permissive: daily rebalancing, high drawdown limit, high turnover cap), run the engine, and compare its wealth curve to two benchmarks: a **risk-free** (constant growth) and a **buy-and-hold equal-weight** portfolio.

> **What should you see?** Three wealth curves. The rebalancing engine should adapt to market conditions — reducing risk exposure during bearish periods (positive lambda) and increasing it during bullish ones. Whether it beats the buy-and-hold depends on how well the EMA signal tracks actual regime shifts in the synthetic data.

In [ ]:
let
    # Define permissive trigger rules (we'll tighten these in Example 2) -
    rules = build(MyTriggerRules, (
        max_drawdown = 0.50,                          # 50% — very permissive for now
        max_turnover = 1.0,                            # 100% — no cap for now
        rebalance_schedule = ones(Int, n_trading_days) # daily rebalancing
    ));

    # Run the rebalancing engine -
    global engine_results = run_rebalancing_engine(context, rules, lambda_series; offset=offset);

    # Compute engine wealth series -
    global engine_wealth = compute_wealth_series(engine_results, price_matrix, tickers; offset=offset);

    # Benchmark 1: risk-free (4.3% annual) -
    risk_free_rate = 0.043;
    global riskfree_wealth = [B₀ * exp(risk_free_rate * (d - 1) * Δt) for d in 1:(n_trading_days + 1)];

    # Benchmark 2: buy-and-hold equal-weight -
    K = length(tickers);
    budget_per_ticker = B₀ / K;
    initial_shares = [budget_per_ticker / price_matrix[offset, k + 1] for k in 1:K];
    global buyhold_wealth = zeros(n_trading_days + 1);
    for d in 0:n_trading_days
        actual_day = offset + d;
        buyhold_wealth[d + 1] = sum(initial_shares[k] * price_matrix[actual_day, k + 1] for k in 1:K);
    end

    println("Engine final wealth:     \$$(round(engine_wealth[end], digits=2))")
    println("Risk-free final wealth:  \$$(round(riskfree_wealth[end], digits=2))")
    println("Buy-and-hold final:      \$$(round(buyhold_wealth[end], digits=2))")
end

**Visualize:** Wealth curves — AI rebalancing engine vs. risk-free vs. buy-and-hold.

> **What should you see?** The risk-free curve is a smooth exponential. The buy-and-hold tracks market movements passively. The engine's curve should show periods where it diverges from buy-and-hold — pulling back during bearish sentiment and pushing forward during bullish sentiment.

In [ ]:
let
    days = 1:(n_trading_days + 1);

    plot(days, engine_wealth, label="AI Rebalancing Engine", linewidth=2.5, color=:steelblue)
    plot!(days, buyhold_wealth, label="Buy-and-Hold (equal-weight)", linewidth=2, color=:coral, linestyle=:dash)
    plot!(days, riskfree_wealth, label="Risk-Free (4.3%)", linewidth=1.5, color=:gray50, linestyle=:dot)
    xlabel!("Trading Day (after warmup)")
    ylabel!("Wealth (\$)")
    title!("AI Rebalancing Engine vs. Benchmarks")
    plot!(size=(800, 450), legend=:topleft)

    # save data for Example 2 -
    save(joinpath(_PATH_TO_DATA, "engine-run-data.jld2"),
        Dict("engine_results" => engine_results, "engine_wealth" => engine_wealth,
             "buyhold_wealth" => buyhold_wealth, "riskfree_wealth" => riskfree_wealth,
             "price_matrix" => price_matrix, "lambda_series" => lambda_series,
             "market_prices" => market_prices, "gm_ema" => gm_ema,
             "tickers" => tickers, "sim_params" => sim_params,
             "context" => context));
end

___
## Summary
In this example, we built the complete AI rebalancing engine pipeline: synthetic market data → EMA crossover → lambda sentiment signal → SIM-based allocation → daily rebalancing loop. The engine adapts its risk exposure in real time based on market conditions. In the next example, we'll tighten the trigger rules and produce a formal scorecard comparison.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.